# Cart2Insights — Statistical Analysis

## Objective

The objective of this analysis is to formally evaluate important relationships identified during EDA.

The project requires three statistical tests:

1. Independent Two-Sample T-Test
   - Delivery Status vs Review Score

2. One-Way ANOVA
   - Product Category vs Order Value

3. Chi-Square Test of Independence
   - Payment Method vs Order Status



In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

In [2]:
PROJECT_ROOT = Path.cwd().parent

FEATURES_DIR = PROJECT_ROOT / "data" / "features"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"

print("Project root:", PROJECT_ROOT)

Project root: e:\GUVI Projects\Cart2Insights_Project1\Cart2Insights


In [3]:
order_features = pd.read_csv(
    FEATURES_DIR / "order_features.csv"
)

customer_features = pd.read_csv(
    FEATURES_DIR / "customer_features.csv"
)

seller_features = pd.read_csv(
    FEATURES_DIR / "seller_features.csv"
)

orders = pd.read_csv(
    CLEANED_DIR / "orders.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)

order_reviews = pd.read_csv(
    CLEANED_DIR / "order_reviews.csv",
    parse_dates=[
        "review_creation_date",
        "review_answer_timestamp",
    ],
)

order_items = pd.read_csv(
    CLEANED_DIR / "order_items.csv",
    parse_dates=["shipping_limit_date"],
)

products = pd.read_csv(
    CLEANED_DIR / "products.csv"
)

order_payments = pd.read_csv(
    CLEANED_DIR / "order_payments.csv"
)

# 1. Independent Two-Sample T-Test

## Business Question

Do delayed orders receive lower review scores than orders delivered on time?

## Hypotheses

### Null Hypothesis (Ho)

The mean review score is the same for on-time and delayed orders.

### Alternative Hypothesis (H₁)

The mean review score differs between on-time and delayed orders.

## Variables

- Grouping variable: `delivery_status`
- Outcome variable: order-level `review_score`

## Why this test?

There are two independent groups:

- On Time
- Delayed

and we want to compare their mean review scores.

Therefore, an independent two-sample t-test is appropriate.

## Analytical Grain

One row represents one order.

When an order has multiple review records, the review scores are averaged so that each order contributes one observation.

In [4]:
order_review_scores = (
    order_reviews
    .groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean")
    )
)

order_review_scores.head()

,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0


In [5]:
delivery_review = (
    order_features[
        ["order_id", "delivery_status"]
    ]
    .merge(
        order_review_scores,
        on="order_id",
        how="inner",
        validate="one_to_one"
    )
    .dropna(
        subset=["delivery_status", "review_score"]
    )
)

delivery_review.head()

,order_id,delivery_status,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,On Time,5.0
1,00018f77f2f0320c557190d7a144bdd3,On Time,4.0
2,000229ec398224ef6ca0657da4fc703e,On Time,5.0
3,00024acbcdf0a6daa1e931b038114c75,On Time,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,On Time,5.0


In [6]:
assert delivery_review["order_id"].is_unique

print("Rows:", len(delivery_review))
print("Unique orders:", delivery_review["order_id"].nunique())

Rows: 95830
Unique orders: 95830


In [7]:
on_time_scores = delivery_review.loc[
    delivery_review["delivery_status"] == "On Time",
    "review_score"
]

delayed_scores = delivery_review.loc[
    delivery_review["delivery_status"] == "Delayed",
    "review_score"
]

In [8]:
print(
    f"On Time: n={len(on_time_scores):,}, "
    f"mean={on_time_scores.mean():.3f}, "
    f"median={on_time_scores.median():.3f}, "
    f"std={on_time_scores.std():.3f}"
)

print(
    f"Delayed: n={len(delayed_scores):,}, "
    f"mean={delayed_scores.mean():.3f}, "
    f"median={delayed_scores.median():.3f}, "
    f"std={delayed_scores.std():.3f}"
)

On Time: n=88,168, mean=4.294, median=5.000, std=1.146
Delayed: n=7,662, mean=2.567, median=2.000, std=1.658


In [9]:
mean_on_time = on_time_scores.mean()
mean_delayed = delayed_scores.mean()

mean_difference = mean_on_time - mean_delayed

print(f"On-time mean: {mean_on_time:.3f}")
print(f"Delayed mean: {mean_delayed:.3f}")
print(f"Mean difference: {mean_difference:.3f}")

On-time mean: 4.294
Delayed mean: 2.567
Mean difference: 1.728


In [10]:
t_statistic, p_value = stats.ttest_ind(
    on_time_scores,
    delayed_scores,
    equal_var=False
)

print(f"T-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.10f}")

T-statistic: 89.3845
P-value: 0.0000000000


In [11]:
alpha = 0.05

if p_value < alpha:
    print("Reject H0")
else:
    print("Fail to reject H0")

Reject H0


In [12]:

print(f"P-value: {p_value:.3e}")

P-value: 0.000e+00


## T-Test — Final Result

### Result

- On-time orders: mean review score = 4.294
- Delayed orders: mean review score = 2.567
- Mean difference = 1.728 points
- T-statistic = 89.3845
- P-value < 0.05
- Decision: Reject H0

### Statistical Interpretation

There is statistically significant evidence that mean review scores differ between on-time and delayed orders.

### Practical Interpretation

The observed difference is 1.728 review-score points on a 1–5 scale, indicating a substantial difference in customer satisfaction between the two delivery-status groups.

### Business Insight

Delayed orders are strongly associated with lower customer review scores. Delivery reliability therefore appears to be an important customer-experience area.

### Business Recommendation

The platform should investigate the causes of delivery delays and prioritize reducing missed estimated delivery dates, particularly in locations or operational segments with higher delay rates.



# 2. One-Way ANOVA

## Business Question

Does order value differ significantly across product categories?

## Null Hypothesis (Ho)

All product categories have the same mean order value.

## Alternative Hypothesis (H₁)

At least one product category has a different mean order value.

## Variables

- Grouping variable: `product_category_name`
- Numerical outcome: `category_order_value`

## Why this test?

Product category contains more than two groups, and we want to compare their mean order values. Therefore, a one-way ANOVA is appropriate.

## Analytical Grain

One row represents one order-category combination.

For orders containing products from multiple categories, only the value contributed by that category is assigned to the category.

In [13]:
category_order_value = (
    order_items[["order_id", "product_id", "price", "freight_value"]]
    .merge(
        products[["product_id", "product_category_name"]],
        on="product_id",
        how="left",
        validate="many_to_one",
    )
    .assign(item_total_value=lambda df: df["price"] + df["freight_value"])
    .groupby(["order_id", "product_category_name"], as_index=False)
    .agg(category_order_value=("item_total_value", "sum"))
)

In [14]:
category_order_value.head()

,order_id,product_category_name,category_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff,72.19
1,00018f77f2f0320c557190d7a144bdd3,pet_shop,259.83
2,000229ec398224ef6ca0657da4fc703e,moveis_decoracao,216.87
3,00024acbcdf0a6daa1e931b038114c75,perfumaria,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,ferramentas_jardim,218.04


In [15]:
print(
    "Rows:",
    len(category_order_value)
)

print(
    "Categories:",
    category_order_value["product_category_name"]
    .nunique()
)

Rows: 98019
Categories: 73


In [16]:
assert category_order_value[
    ["order_id", "product_category_name"]
].duplicated().sum() == 0

print("One row per order-category validated.")

One row per order-category validated.


In [17]:
category_counts = (
    category_order_value["product_category_name"]
    .value_counts()
)

category_counts.describe()

count      73.000000
mean     1342.726027
std      2252.345988
min         2.000000
25%        75.000000
50%       244.000000
75%      1273.000000
max      9417.000000
Name: count, dtype: float64

In [18]:
valid_categories = category_counts[
    category_counts >= 100
].index

anova_data = category_order_value[
    category_order_value["product_category_name"]
    .isin(valid_categories)
].copy()

print(
    "Categories included:",
    anova_data["product_category_name"].nunique()
)

print(
    "Observations included:",
    len(anova_data)
)

Categories included: 51
Observations included: 97155


In [19]:
anova_summary = (
    anova_data.groupby("product_category_name")["category_order_value"]
    .agg(count="count", mean="mean", median="median", std="std")
    .sort_values("mean", ascending=False)
)

anova_summary

,count,mean,median,std
product_category_name,,,,
pcs,181,1286.184696,1250.810,736.948613
eletrodomesticos_2,234,529.563761,283.270,559.861999
agro_industria_e_comercio,182,430.626758,410.615,616.049081
instrumentos_musicais,628,334.613646,141.205,484.208388
eletroportateis,630,328.045762,128.835,506.770414
telefonia_fixa,217,295.948433,74.590,1096.710735
moveis_escritorio,1273,269.075137,206.820,215.090607
construcao_ferramentas_seguranca,167,266.249222,165.620,375.737907
climatizacao,253,244.166759,164.090,271.020699


In [20]:
anova_groups = [
    group["category_order_value"].values
    for _, group in anova_data.groupby(
        "product_category_name"
    )
]

print(
    "Number of groups:",
    len(anova_groups)
)

Number of groups: 51


In [21]:
f_statistic, anova_p_value = stats.f_oneway(
    *anova_groups
)

print(
    f"F-statistic: {f_statistic:.4f}"
)

print(
    f"P-value: {anova_p_value:.10f}"
)

F-statistic: 209.1657
P-value: 0.0000000000


In [22]:
alpha = 0.05

if anova_p_value < alpha:
    print("Reject H0")
else:
    print("Fail to reject H0")

Reject H0


In [23]:
anova_variances = (
    anova_data
    .groupby("product_category_name")
    ["category_order_value"]
    .var()
    .sort_values(ascending=False)
)

anova_variances

product_category_name
telefonia_fixa                                    1.202774e+06
pcs                                               5.430933e+05
agro_industria_e_comercio                         3.795165e+05
eletrodomesticos_2                                3.134455e+05
eletroportateis                                   2.568163e+05
instrumentos_musicais                             2.344578e+05
artes                                             2.193084e+05
construcao_ferramentas_seguranca                  1.411790e+05
sinalizacao_e_seguranca                           1.409300e+05
construcao_ferramentas_construcao                 9.565940e+04
industria_comercio_e_negocios                     8.487579e+04
climatizacao                                      7.345222e+04
relogios_presentes                                7.340708e+04
consoles_games                                    6.360473e+04
automotivo                                        5.999342e+04
ferramentas_jardim               

In [24]:
anova_summary.head(15)

,count,mean,median,std
product_category_name,,,,
pcs,181,1286.184696,1250.810,736.948613
eletrodomesticos_2,234,529.563761,283.270,559.861999
agro_industria_e_comercio,182,430.626758,410.615,616.049081
instrumentos_musicais,628,334.613646,141.205,484.208388
eletroportateis,630,328.045762,128.835,506.770414
telefonia_fixa,217,295.948433,74.590,1096.710735
moveis_escritorio,1273,269.075137,206.820,215.090607
construcao_ferramentas_seguranca,167,266.249222,165.620,375.737907
climatizacao,253,244.166759,164.090,271.020699


## ANOVA — Final Result

### Hypotheses

**Null Hypothesis (H0):**  
All product categories have the same mean order value.

**Alternative Hypothesis (H₁):**  
At least one product category has a different mean order value.

### Test Result

- **F-statistic:** 209.1657
- **P-value:** < 0.05
- **Decision:** Reject H0

### Statistical Interpretation

The one-way ANOVA provides statistically significant evidence that mean order value differs across product categories.

Therefore, we reject the null hypothesis that all product categories have the same mean order value.

### Business Interpretation

Product category is associated with differences in order value. This indicates that the category mix is relevant when analyzing customer purchasing behavior and revenue performance.

### Business Impact

Categories associated with higher order values may provide opportunities for premium product merchandising, targeted promotions, and revenue optimization.



# 3. Chi-Square Test of Independence

## Business Question

Is payment method associated with order status?

## Hypotheses

### Null Hypothesis (H0)

Payment method and order status are independent.

### Alternative Hypothesis (H₁)

Payment method and order status are associated.

## Variables

- Categorical variable 1: Primary Payment Method
- Categorical variable 2: Order Status

## Why this test?

Both variables are categorical, and the objective is to determine whether they are statistically associated. Therefore, a chi-square test of independence is appropriate.

## Analytical Grain

One row per order.

Because an order may have multiple payment records, the primary payment method is defined as the payment type contributing the largest total payment value within the order.

In [25]:
order_payment_type = (
    order_payments
    .groupby(
        ["order_id", "payment_type"],
        as_index=False
    )
    .agg(
        payment_value=("payment_value", "sum")
    )
    .sort_values(
        ["order_id", "payment_value"],
        ascending=[True, False]
    )
)

In [26]:
primary_payment = (
    order_payment_type
    .drop_duplicates("order_id")
    [["order_id", "payment_type"]]
    .rename(
        columns={
            "payment_type": "primary_payment_type"
        }
    )
)

assert primary_payment["order_id"].is_unique

primary_payment.head()

,order_id,primary_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,credit_card
1,00018f77f2f0320c557190d7a144bdd3,credit_card
2,000229ec398224ef6ca0657da4fc703e,credit_card
3,00024acbcdf0a6daa1e931b038114c75,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,credit_card


In [27]:
payment_status = (
    orders[
        ["order_id", "order_status"]
    ]
    .merge(
        primary_payment,
        on="order_id",
        how="inner",
        validate="one_to_one"
    )
)

payment_status.head()

,order_id,order_status,primary_payment_type
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,voucher
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,boleto
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,credit_card
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,credit_card
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,credit_card


In [28]:
print("Rows:", len(payment_status))
print(
    "Unique orders:",
    payment_status["order_id"].nunique()
)

assert payment_status["order_id"].is_unique

Rows: 99440
Unique orders: 99440


In [29]:
contingency_table = pd.crosstab(
    payment_status["primary_payment_type"],
    payment_status["order_status"]
)

contingency_table

order_status,approved,canceled,created,delivered,invoiced,processing,shipped,unavailable
primary_payment_type,,,,,,,,
boleto,0,95,2,19191,67,70,209,150
credit_card,2,432,3,72785,231,219,836,427
debit_card,0,7,0,1484,6,2,22,6
not_defined,0,3,0,0,0,0,0,0
voucher,0,88,0,3017,10,10,40,26


In [30]:
chi2_statistic, chi2_p_value, degrees_of_freedom, expected = stats.chi2_contingency(
    contingency_table
)

print(f"Chi-square statistic: {chi2_statistic:.4f}")

print(f"Degrees of freedom: {degrees_of_freedom}")

print(f"P-value: {chi2_p_value:.10f}")

Chi-square statistic: 738.7755
Degrees of freedom: 28
P-value: 0.0000000000


In [31]:
alpha = 0.05

if chi2_p_value < alpha:
    print("Reject H0")
else:
    print("Fail to reject H0")

Reject H0


In [32]:
expected_table = pd.DataFrame(
    expected,
    index=contingency_table.index,
    columns=contingency_table.columns
)

expected_table

order_status,approved,canceled,created,delivered,invoiced,processing,shipped,unavailable
primary_payment_type,,,,,,,,
boleto,0.397908,124.346340,0.994771,19194.498874,62.471601,59.885197,220.242237,121.163073
credit_card,1.507140,470.981245,3.767850,72702.172114,236.620977,226.824568,834.201981,458.924125
debit_card,0.030712,9.597496,0.076780,1481.500191,4.821782,4.622154,16.999085,9.351800
not_defined,0.000060,0.018856,0.000151,2.910609,0.009473,0.009081,0.033397,0.018373
voucher,0.064179,20.056064,0.160449,3095.918212,10.076167,9.659000,35.523300,19.542629


In [33]:
print(
    "Minimum expected frequency:",
    expected_table.to_numpy().min()
)

Minimum expected frequency: 6.033789219629928e-05


## Chi-Square Test — Final Result

### Business Question

Is primary payment method associated with order status?

### Hypotheses

**Null Hypothesis (H0):**  
Primary payment method and order status are independent.

**Alternative Hypothesis (H₁):**  
Primary payment method and order status are associated.

### Test Result

- **Chi-square statistic:** 738.7755
- **Degrees of freedom:** 28
- **P-value:** < 0.05
- **Decision:** Reject H0

### Statistical Interpretation

The chi-square test indicates statistically significant evidence of an association between primary payment method and order status.

Therefore, we reject the null hypothesis that payment method and order status are independent.

### Business Interpretation

Order-status patterns differ across payment methods. This suggests that payment method may be relevant when investigating order completion and cancellation patterns.

### Business Impact

Payment methods with unusual order-status patterns can be investigated further for payment-processing, checkout, or transaction-related issues.

### Important Limitation

Some expected cell frequencies in the contingency table were extremely small. Therefore, the standard chi-square approximation and its p-value should be interpreted with caution.



# Statistical Analysis — Final Summary

The statistical analysis evaluated three business questions identified during EDA.

## 1. Delivery Status vs Review Score — Independent Two-Sample T-Test

The analysis compared review scores between on-time and delayed orders.

- On-time mean review score: **4.294**
- Delayed mean review score: **2.567**
- Mean difference: **1.728 points**
- T-statistic: **89.3845**
- Decision: **Reject H0**

### Conclusion

There is statistically significant evidence that review scores differ between on-time and delayed orders. Delayed orders have substantially lower average review scores than on-time orders.

This indicates a strong association between delivery performance and customer satisfaction.

---

## 2. Product Category vs Order Value — One-Way ANOVA

The analysis evaluated whether mean order value differs across product categories.

- F-statistic: **209.1657**
- Decision: **Reject H0**

### Conclusion

There is statistically significant evidence that mean order value differs across product categories.

This indicates that product category is associated with differences in order value.

---

## 3. Payment Method vs Order Status — Chi-Square Test of Independence

The analysis evaluated whether primary payment method and order status are independent.

- Chi-square statistic: **738.7755**
- Degrees of freedom: **28**
- Decision: **Reject H0**

### Conclusion

The chi-square analysis indicates statistically significant evidence of an association between primary payment method and order status.

However, some expected cell frequencies were extremely small in the contingency table. Therefore, the standard chi-square approximation and its p-value should be interpreted with caution.

---

# Overall Business Interpretation

The statistical analysis supports three important areas for further business attention:

1. **Delivery performance and customer satisfaction** are strongly associated, with delayed orders receiving substantially lower review scores.
2. **Product category and order value** are associated, indicating that category mix is relevant to purchasing value.
3. **Payment method and order status** show an observed association, although the chi-square result requires caution because of sparse expected frequencies.

## Important Statistical Principle

Statistical significance indicates evidence against the null hypothesis. It does not automatically imply practical importance, and these observational analyses do not establish causation.

The statistical results should therefore be interpreted together with the EDA findings and business context.